# 手势识别控制系统

## 项目简介

本项目实现一个由 PC 或树莓派负责手势识别、Arduino 负责外设控制的手势识别控制系统。视觉端识别用户手势后，通过串口把手势标签和置信度发给 Arduino，Arduino 将手势映射为双轴云台、继电器设备和状态指示灯的动作，实现“识别即控制”的交互原型。


## 主要器件

| 器件 | 数量 | 说明 |
| --- | --- | --- |
| PC 或树莓派 | 1 | 运行手势识别算法 |
| 摄像头 | 1 | 采集手部图像 |
| Arduino Uno | 1 | 接收手势并控制执行端 |
| 舵机 | 2 | 控制双轴云台 |
| 继电器 | 1 | 模拟家电开关 |
| LED | 2 | 状态指示 |


## 核心功能

- 视觉端识别左右、上下、开关、停止等手势。
- Arduino 根据手势驱动云台水平和俯仰角移动。
- `OPEN / CLOSE` 手势可直接控制继电器状态。
- 低置信度手势会被忽略，降低误触发概率。
- 长时间没有新手势时自动回到等待识别指示状态。


## 引脚分配

| 模块 | 引脚 |
| --- | --- |
| 水平舵机 | D9 |
| 俯仰舵机 | D10 |
| 绿色指示灯 | D5 |
| 红色指示灯 | D6 |
| 继电器 | D8 |
| 视觉端串口通信 | USB Serial |


## 接线说明

- 两路舵机分别接 D9 和 D10，建议使用独立 5V 供电并与 Arduino 共地。
- 继电器接 D8，可接灯带、风扇或其它低压演示负载。
- 绿色和红色 LED 分别用于表示“已识别动作”和“空闲等待”状态。
- 视觉端通过 USB 串口向 Arduino 发送手势识别结果，波特率为 `115200`。


## 串口协议

- 视觉端发送格式：`G,gesture,confidence`。
- 示例：`G,LEFT,84`、`G,OPEN,92`。
- 仅当置信度大于等于 `MIN_CONFIDENCE` 时，Arduino 才执行对应动作。
- Arduino 会返回 `ACK,gesture,confidence`，便于上位机记录执行结果。


## 运行方式

1. 上传 `src/gesture_recognition_control_system/gesture_recognition_control_system.ino` 到 Arduino。
2. 视觉端部署手势识别程序，可使用传统分类器或 MediaPipe 等方案。
3. 将识别结果按协议通过串口发送到 Arduino。
4. 先使用串口助手模拟发送 `LEFT/RIGHT/OPEN/CLOSE`，验证各动作映射是否正确。
5. 根据实际控制需求调整步进角度和最小置信度阈值。


## 控制逻辑说明

- `LEFT / RIGHT / UP / DOWN` 手势控制双轴云台位置。
- `OPEN / CLOSE` 手势控制继电器开关，便于扩展到灯光或风扇控制场景。
- `STOP` 手势会将云台拉回中心位置，作为一种快速复位指令。
- 手势识别结果会记录最后一次标签和置信度，并通过串口回传 ACK。
- 长时间无新命令时，红灯点亮表示系统处于等待识别状态，但不会强制改变当前负载状态。


## 验证标准

| 测试项 | 通过标准 |
| --- | --- |
| 左右控制 | `LEFT/RIGHT` 能驱动水平舵机变化 |
| 上下控制 | `UP/DOWN` 能驱动俯仰舵机变化 |
| 设备开关 | `OPEN/CLOSE` 能改变继电器状态 |
| 置信度过滤 | 低于阈值的命令不会触发动作 |


## 可扩展方向

- 接入机械臂控制，扩展成手势遥操作系统。
- 用 MediaPipe 手部关键点替代简单分类器。
- 增加多用户手势集和自定义动作学习功能。
- 把串口链路升级为蓝牙或 WiFi 无线交互。
